# Evaluating LLM Models for the Text-to-SQL Task
In this notebook, we will evaluate the performance of the **Qwen2.5-Coder-7B-Instruct** model in the Text-to-SQL task. 
We will use the **Spider** dataset, evaluating the correctness of the generated queries via **Execution Accuracy** (comparing the real execution results on the database).

## 1. Environment Setup
We start by checking the available hardware resources (GPU) and installing the necessary libraries to manage the LLM via HuggingFace: `transformers/accelerate` to download and run the model, `bitsandbytes` for 4-bit quantization, `datasets` to download the Spider dataset.

In [17]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA disponibile:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA disponibile: True
GPU: Tesla T4
CUDA: 12.8


In [19]:
!pip install -q -U transformers accelerate bitsandbytes datasets sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 77.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 43.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.5 MB/s eta 0:00:00


In [20]:
import torch
import transformers
import datasets
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Transformers: 5.15.0
Datasets: 5.0.1
Accelerate: 1.14.0
CUDA: True
GPU: Tesla T4


## 2. Spider Dataset Loading and Analysis
We load the `xlangai/spider` dataset. This dataset consists of pairs of natural language questions and their corresponding SQL queries (Gold SQL).
Since we need the actual databases (SQLite) and table definitions for evaluation, we will also map the local files mounted in the Kaggle environment to load the schemas (`tables.json`) and the validation data (`dev.json`).

In [21]:
from datasets import load_dataset

dataset = load_dataset("xlangai/spider")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})


In [22]:
example = dataset["validation"][0]

print("Database ID:")
print(example["db_id"])

print("\nQuestion:")
print(example["question"])

print("\nGold SQL:")
print(example["query"])

Database ID:
concert_singer

Question:
How many singers do we have?

Gold SQL:
SELECT count(*) FROM singer


We load the dataset directly from Kaggle as input to obtain all the databases, complete with tables and columns.

In [23]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    #for file in files:
        #print(f"{indent}  {file}")

input/
  datasets/
    jeromeblanchet/
      yale-universitys-spider-10-nlp-dataset/
        spider/
          database/
            solvency_ii/
            voter_1/
            concert_singer/
            apartment_rentals/
            flight_1/
            gymnast/
            epinions_1/
            local_govt_and_lot/
            assets_maintenance/
            customers_and_products_contacts/
            restaurants/
            insurance_policies/
            local_govt_mdm/
            county_public_safety/
            restaurant_1/
            ship_mission/
            culture_company/
            city_record/
            inn_1/
              data_csv/
            yelp/
            insurance_and_eClaims/
            tracking_grants_for_research/
            machine_repair/
            mountain_photos/
            storm_record/
            wedding/
            train_station/
            party_host/
            csu_1/
            company_employee/
            entertainment_award

In [24]:
import os
import json

# Database path
BASE_PATH = "/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider"

DEV_PATH = os.path.join(BASE_PATH, "dev.json")
TABLES_PATH = os.path.join(BASE_PATH, "tables.json")
DATABASE_PATH = os.path.join(BASE_PATH, "database")

with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

print("Number of examples in the Dev set:", len(dev_data))

print(json.dumps(dev_data[0], indent=2, ensure_ascii=False))

Numero di esempi nel Dev set: 1034
{
  "db_id": "concert_singer",
  "query": "SELECT count(*) FROM singer",
  "query_toks": [
    "SELECT",
    "count",
    "(",
    "*",
    ")",
    "FROM",
    "singer"
  ],
  "query_toks_no_value": [
    "select",
    "count",
    "(",
    "*",
    ")",
    "from",
    "singer"
  ],
  "question": "How many singers do we have?",
  "question_toks": [
    "How",
    "many",
    "singers",
    "do",
    "we",
    "have",
    "?"
  ],
  "sql": {
    "except": null,
    "from": {
      "conds": [],
      "table_units": [
        [
          "table_unit",
          1
        ]
      ]
    },
    "groupBy": [],
    "having": [],
    "intersect": null,
    "limit": null,
    "orderBy": [],
    "select": [
      false,
      [
        [
          3,
          [
            0,
            [
              0,
              0,
              false
            ],
            null
          ]
        ]
      ]
    ],
    "union": null,
    "where": []
  }
}


In [25]:
# tables_data contains all the database schemas (table names, columns, keys)
TABLES_PATH = os.path.join(BASE_PATH, "tables.json")

with open(TABLES_PATH, "r", encoding="utf-8") as f:
    tables_data = json.load(f)

print("Numebr of databases in tables.json:", len(tables_data))

print("\nFirst element:")
print(json.dumps(tables_data[0], indent=2, ensure_ascii=False))

Numero di database in tables.json: 166

Primo elemento:
{
  "column_names": [
    [
      -1,
      "*"
    ],
    [
      0,
      "perpetrator id"
    ],
    [
      0,
      "people id"
    ],
    [
      0,
      "date"
    ],
    [
      0,
      "year"
    ],
    [
      0,
      "location"
    ],
    [
      0,
      "country"
    ],
    [
      0,
      "killed"
    ],
    [
      0,
      "injured"
    ],
    [
      1,
      "people id"
    ],
    [
      1,
      "name"
    ],
    [
      1,
      "height"
    ],
    [
      1,
      "weight"
    ],
    [
      1,
      "home town"
    ]
  ],
  "column_names_original": [
    [
      -1,
      "*"
    ],
    [
      0,
      "Perpetrator_ID"
    ],
    [
      0,
      "People_ID"
    ],
    [
      0,
      "Date"
    ],
    [
      0,
      "Year"
    ],
    [
      0,
      "Location"
    ],
    [
      0,
      "Country"
    ],
    [
      0,
      "Killed"
    ],
    [
      0,
      "Injured"
    ],
    [
      1,
     

## 3. Building the Database Schema for the Prompt
To generate accurate SQL, LLMs need to understand the database structure.
In this section, we create the `get_database_schema` function which, starting from the `tables.json` file, extracts a formatted text representation for each database containing:
*   Table names
*   Column names and data types
*   Primary Keys and Foreign Keys

In [26]:
# Dictionary for quick schema access using the db_id
schemas = {item["db_id"]: item for item in tables_data}

def get_database_schema(db_id):
    
    if db_id not in schemas:
        raise ValueError(f"Database '{db_id}' not found in tables.json")
    
    db_schema = schemas[db_id]
    
    table_names = db_schema["table_names_original"]
    column_names = db_schema["column_names_original"]
    column_types = db_schema["column_types"]
    primary_keys = db_schema["primary_keys"]
    foreign_keys = db_schema["foreign_keys"]
    
    # Construct the description of tables
    schema_text = []
    
    for table_idx, table_name in enumerate(table_names):
        schema_text.append(f"Table: {table_name}")
        
        # Search the columns of this table
        for col_idx, (col_table_idx, col_name) in enumerate(column_names):
            
            # -1 indicates the special column "*"
            if col_table_idx == table_idx:
                col_type = column_types[col_idx]
                
                # Check if the column is a primary key, appending a label
                pk_label = " [PRIMARY KEY]" if col_idx in primary_keys else ""
                
                schema_text.append(
                    f"  - {col_name} ({col_type}){pk_label}"
                )
        
        schema_text.append("")
    
    # Foreign keys
    if foreign_keys:
        schema_text.append("Foreign Keys:")
        
        for fk_from, fk_to in foreign_keys:
            
            from_table_idx, from_column = column_names[fk_from]
            to_table_idx, to_column = column_names[fk_to]
            
            from_table = table_names[from_table_idx]
            to_table = table_names[to_table_idx]
            
            schema_text.append(
                f"  - {from_table}.{from_column} → "
                f"{to_table}.{to_column}"
            )
    
    return "\n".join(schema_text)

print(get_database_schema("concert_singer"))

Table: stadium
  - Stadium_ID (number) [PRIMARY KEY]
  - Location (text)
  - Name (text)
  - Capacity (number)
  - Highest (number)
  - Lowest (number)
  - Average (number)

Table: singer
  - Singer_ID (number) [PRIMARY KEY]
  - Name (text)
  - Country (text)
  - Song_Name (text)
  - Song_release_year (text)
  - Age (number)
  - Is_male (others)

Table: concert
  - concert_ID (number) [PRIMARY KEY]
  - concert_Name (text)
  - Theme (text)
  - Stadium_ID (text)
  - Year (text)

Table: singer_in_concert
  - concert_ID (number) [PRIMARY KEY]
  - Singer_ID (text)

Foreign Keys:
  - concert.Stadium_ID → stadium.Stadium_ID
  - singer_in_concert.Singer_ID → singer.Singer_ID
  - singer_in_concert.concert_ID → concert.concert_ID


## 4. Model Loading (Qwen2.5-Coder-7B)
We use **Qwen/Qwen2.5-Coder-7B-Instruct**, an excellent open-weights model for programming and Text-to-SQL tasks.
To run it smoothly on the T4 GPU (which has 16GB of VRAM), we apply **4-bit quantization** via NF4 (NormalFloat4), while keeping the computational dtype at Float16 to prevent precision loss during inference.

In [27]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [28]:
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

model.eval()

print("Model loaded successfully!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [29]:
print(model.hf_device_map)

{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 1, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


## 5. Zero-Shot Prompting
In the first experiment, we provide the model with only the **System Prompt**, the **DB Schema**, and the **Question**, without any practical examples.
We also build a cleanup function (`clean_sql`) to remove any Markdown fences generated by the model.

To evaluate the model, we won't rely on simple *Exact Match* (string comparison), but rather **Execution Accuracy**: we will execute both the Gold query and the Predicted query on the actual SQLite database and mark the prediction as correct only if the returned result sets are identical.

In [30]:
def build_zero_shot_prompt(question, schema):
    prompt = f"""You are an expert Text-to-SQL system.

Your task is to convert a natural language question into a SQL query.

Use only the tables and columns provided in the database schema.
Do not invent tables or columns.

Database schema:
{schema}

Question:
{question}

Return only the SQL query, without explanations, markdown or code fences.
"""
    
    return prompt

In [31]:
example = dev_data[0]

question = example["question"]
db_id = example["db_id"]
gold_sql = example["query"]

schema = get_database_schema(db_id)

prompt = build_zero_shot_prompt(question, schema)

print(prompt)

You are an expert Text-to-SQL system.

Your task is to convert a natural language question into a SQL query.

Use only the tables and columns provided in the database schema.
Do not invent tables or columns.

Database schema:
Table: stadium
  - Stadium_ID (number) [PRIMARY KEY]
  - Location (text)
  - Name (text)
  - Capacity (number)
  - Highest (number)
  - Lowest (number)
  - Average (number)

Table: singer
  - Singer_ID (number) [PRIMARY KEY]
  - Name (text)
  - Country (text)
  - Song_Name (text)
  - Song_release_year (text)
  - Age (number)
  - Is_male (others)

Table: concert
  - concert_ID (number) [PRIMARY KEY]
  - concert_Name (text)
  - Theme (text)
  - Stadium_ID (text)
  - Year (text)

Table: singer_in_concert
  - concert_ID (number) [PRIMARY KEY]
  - Singer_ID (text)

Foreign Keys:
  - concert.Stadium_ID → stadium.Stadium_ID
  - singer_in_concert.Singer_ID → singer.Singer_ID
  - singer_in_concert.concert_ID → concert.concert_ID

Question:
How many singers do we have?

Ret

We connect the prompt to the model.

In [32]:
def generate_sql(prompt, max_new_tokens=64):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    # We move the inputs to the model's device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False, # make the generation deterministsic (Greedy Search)
            #temperature=0.0
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return generated_text.strip()

def clean_sql(sql):
    sql = sql.strip()

    if sql.startswith("```sql"):
        sql = sql[6:]

    elif sql.startswith("```"):
        sql = sql[3:]

    if sql.endswith("```"):
        sql = sql[:-3]

    return sql.strip()

In [33]:
predicted_sql = generate_sql(prompt)
predicted_sql = clean_sql(predicted_sql)

print("Question:")
print(question)

print("\nGold SQL:")
print(gold_sql)

print("\nGenerated SQL:")
print(predicted_sql)

Question:
How many singers do we have?

Gold SQL:
SELECT count(*) FROM singer

Generated SQL:
SELECT COUNT(*) FROM singer;


Now, we evaluate a few examples, rather than all 1034, by comparing the correct query with the generated query using the Execution Accuracy function.

In [34]:
import sqlite3

In [35]:
def get_database_path(db_id):
    return os.path.join(
        DATABASE_PATH,
        db_id,
        f"{db_id}.sqlite"
    )


db_path = get_database_path("concert_singer")

print(db_path)
print("Exists:", os.path.exists(db_path))

/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database/concert_singer/concert_singer.sqlite
Exists: True


In [36]:
# Function to execute a SQL query 
def execute_sql(db_id, sql):
    db_path = get_database_path(db_id)

    conn = sqlite3.connect(db_path)

    try:
        cursor = conn.cursor()
        cursor.execute(sql)
        result = cursor.fetchall()
        return result

    except Exception as e:
        return None

    finally:
        conn.close()


# Comparison function
def execution_accuracy(db_id, gold_sql, predicted_sql):
    gold_result = execute_sql(db_id, gold_sql)
    predicted_result = execute_sql(db_id, predicted_sql)

    # If the predicted query is not executable
    if predicted_result is None:
        return False

    # Compare the results
    return gold_result == predicted_result

In [24]:
example = dev_data[0]

db_id = example["db_id"]
gold_sql = example["query"]

predicted_sql = clean_sql(generate_sql(prompt))

print("Database:", db_id)
print("Gold SQL:", gold_sql)
print("Predicted SQL:", predicted_sql)

print("\nGold result:")
print(execute_sql(db_id, gold_sql))

print("\nPredicted result:")
print(execute_sql(db_id, predicted_sql))

print("\nExecution correct:")
print(execution_accuracy(db_id, gold_sql, predicted_sql))

Database: concert_singer
Gold SQL: SELECT count(*) FROM singer
Predicted SQL: SELECT COUNT(*) FROM singer;

Gold result:
[(6,)]

Predicted result:
[(6,)]

Execution correct:
True


In [27]:
# We test it on 50 examples
import random

random.seed(42)

test_examples = random.sample(dev_data, 50)

results = []

for i, example in enumerate(test_examples):

    db_id = example["db_id"]
    question = example["question"]
    gold_sql = example["query"]

    schema = get_database_schema(db_id)
    prompt = build_zero_shot_prompt(question, schema)

    predicted_sql = generate_sql(prompt)
    predicted_sql = clean_sql(predicted_sql)

    correct = execution_accuracy(
        db_id,
        gold_sql,
        predicted_sql
    )

    results.append({
        "index": i,
        "db_id": db_id,
        "question": question,
        "gold_sql": gold_sql,
        "predicted_sql": predicted_sql,
        "correct": correct
    })

    print(f"[{i+1}/10] {correct}")

[1/10] False
[2/10] True
[3/10] True
[4/10] True
[5/10] True
[6/10] True
[7/10] True
[8/10] False
[9/10] True
[10/10] False
[11/10] True
[12/10] True
[13/10] False
[14/10] True
[15/10] False
[16/10] True
[17/10] True
[18/10] False
[19/10] False
[20/10] True
[21/10] True
[22/10] True
[23/10] True
[24/10] True
[25/10] True
[26/10] True
[27/10] True
[28/10] True
[29/10] False
[30/10] True
[31/10] True
[32/10] True
[33/10] False
[34/10] True
[35/10] True
[36/10] True
[37/10] True
[38/10] True
[39/10] True
[40/10] True
[41/10] False
[42/10] True
[43/10] False
[44/10] False
[45/10] True
[46/10] True
[47/10] True
[48/10] True
[49/10] True
[50/10] True


In [28]:
# see the errors
errors = [r for r in results if not r["correct"]]

print(f"Numero errori: {len(errors)}")

for error in errors[:10]:
    print("\n--- ERROR ---")
    print("Database:", error["db_id"])
    print("Question:", error["question"])
    print("Gold:", error["gold_sql"])
    print("Predicted:", error["predicted_sql"])
    

accuracy = sum(r["correct"] for r in results) / len(results)

print(f"\nZero-Shot Execution Accuracy: {accuracy:.2%}")

Numero errori: 12

--- ERROR ---
Database: flight_2
Question: Give the code of the airport with the least flights.
Gold: SELECT T1.AirportCode FROM AIRPORTS AS T1 JOIN FLIGHTS AS T2 ON T1.AirportCode  =  T2.DestAirport OR T1.AirportCode  =  T2.SourceAirport GROUP BY T1.AirportCode ORDER BY count(*) LIMIT 1
Predicted: SELECT SourceAirport 
FROM flights 
GROUP BY SourceAirport 
ORDER BY COUNT(*) ASC 
LIMIT 1;

--- ERROR ---
Database: car_1
Question: What are the ids and names of all countries that either have more than 3 car makers or produce fiats?
Gold: SELECT T1.countryId ,  T1.CountryName FROM Countries AS T1 JOIN CAR_MAKERS AS T2 ON T1.CountryId  =  T2.Country GROUP BY T1.countryId HAVING count(*)  >  3 UNION SELECT T1.countryId ,  T1.CountryName FROM Countries AS T1 JOIN CAR_MAKERS AS T2 ON T1.CountryId  =  T2.Country JOIN MODEL_LIST AS T3 ON T2.Id  =  T3.Maker WHERE T3.Model  =  'fiat';
Predicted: SELECT c.CountryId, c.CountryName
FROM countries c
JOIN car_makers cm ON c.CountryId

We execute the **complete zero-shot generation** on all 1034 examples, saving progressively the results in a CSV file.

In [37]:
import pandas as pd
import time

In [96]:
RESULTS_PATH = "/kaggle/working/zero_shot_results.csv"

def save_results(results, path=RESULTS_PATH):
    df = pd.DataFrame(results)
    df.to_csv(path, index=False)
    return df

In [99]:
import os

# Controlla se esiste già un risultato precedente
if os.path.exists(RESULTS_PATH):
    previous_results = pd.read_csv(RESULTS_PATH)
    print(f"Found {len(previous_results)} previously saved results.")
else:
    previous_results = pd.DataFrame()
    print("No previous results found.")

Trovati 30 risultati già salvati.


In [101]:
results = []

start_time = time.time()

for i, example in enumerate(dev_data):

    db_id = example["db_id"]
    question = example["question"]
    gold_sql = example["query"]

    # Construction of the schema
    schema = get_database_schema(db_id)

    # Construction of the zero-shot prompt
    prompt = build_zero_shot_prompt(
        question,
        schema
    )

    # SQL generation
    try:
        predicted_sql = generate_sql(prompt)
        predicted_sql = clean_sql(predicted_sql)

    except Exception as e:
        predicted_sql = ""
        print(f"\nErrore nella generazione dell'esempio {i}: {e}")

    # Execution Accuracy
    correct = execution_accuracy(
        db_id,
        gold_sql,
        predicted_sql
    )

    # Save the result
    results.append({
        "index": i,
        "db_id": db_id,
        "question": question,
        "gold_sql": gold_sql,
        "predicted_sql": predicted_sql,
        "correct": correct
    })

    # Periodic saving
    if (i + 1) % 10 == 0:
        save_results(results)

        elapsed = time.time() - start_time
        accuracy = sum(r["correct"] for r in results) / len(results)

        print(
            f"[{i+1}/{len(dev_data)}] "
            f"Accuracy: {accuracy:.2%} | "
            f"Time: {elapsed/60:.1f} min"
        )

# Final saving
save_results(results)

print("\nGeneration completed.")

[10/1034] Accuracy: 100.00% | Time: 0.3 min
[20/1034] Accuracy: 100.00% | Time: 0.7 min
[30/1034] Accuracy: 93.33% | Time: 1.2 min
[40/1034] Accuracy: 82.50% | Time: 1.8 min
[50/1034] Accuracy: 80.00% | Time: 2.4 min
[60/1034] Accuracy: 78.33% | Time: 3.0 min
[70/1034] Accuracy: 77.14% | Time: 3.6 min
[80/1034] Accuracy: 72.50% | Time: 4.2 min
[90/1034] Accuracy: 74.44% | Time: 4.8 min
[100/1034] Accuracy: 74.00% | Time: 5.4 min
[110/1034] Accuracy: 72.73% | Time: 6.1 min
[120/1034] Accuracy: 68.33% | Time: 6.7 min
[130/1034] Accuracy: 66.15% | Time: 7.1 min
[140/1034] Accuracy: 66.43% | Time: 7.7 min
[150/1034] Accuracy: 67.33% | Time: 8.2 min
[160/1034] Accuracy: 66.25% | Time: 8.8 min
[170/1034] Accuracy: 66.47% | Time: 9.4 min
[180/1034] Accuracy: 63.89% | Time: 10.1 min
[190/1034] Accuracy: 65.79% | Time: 10.3 min
[200/1034] Accuracy: 67.50% | Time: 10.6 min
[210/1034] Accuracy: 69.05% | Time: 10.9 min
[220/1034] Accuracy: 70.45% | Time: 11.6 min
[230/1034] Accuracy: 69.57% | Time

In [102]:
import pandas as pd
import os

RESULTS_PATH = "/kaggle/working/zero_shot_results.csv"

df_saved = pd.read_csv(RESULTS_PATH)

print("Saved results:", len(df_saved))
print("Last ssaved index:", df_saved["index"].max())
print("Total accuracy:", df_saved["correct"].mean())

Risultati salvati: 1034
Ultimo indice salvato: 1033
Accuracy totale: 0.7021276595744681


In [103]:
# check whether the examples were executed and if there were any missed generations or empty SQL statements.
df_zero_shot = pd.read_csv(RESULTS_PATH)

print("Results:", len(df_zero_shot))
print("Correct:", df_zero_shot["correct"].sum())
print("Accuracy:", df_zero_shot["correct"].mean())

df_zero_shot["predicted_sql"].isna().sum()

df_zero_shot[df_zero_shot["correct"] == False].head(10)

Numero risultati: 1034
Corrette: 726
Accuracy: 0.7021276595744681


,index,db_id,question,gold_sql,predicted_sql,correct
22,22,concert_singer,Show the stadium name and the number of concer...,"SELECT T2.name , count(*) FROM concert AS T1 ...","SELECT s.Name, COUNT(c.concert_ID) AS Number_o...",False
23,23,concert_singer,"For each stadium, how many concerts play there?","SELECT T2.name , count(*) FROM concert AS T1 ...","SELECT s.Name, COUNT(c.concert_ID) AS Number_o...",False
30,30,concert_singer,Show countries where a singer above age 40 and...,SELECT country FROM singer WHERE age > 40 IN...,SELECT DISTINCT Country \nFROM singer \nWHERE ...,False
31,31,concert_singer,Show names for all stadiums except for stadium...,SELECT name FROM stadium EXCEPT SELECT T2.name...,SELECT Name FROM stadium WHERE Stadium_ID NOT ...,False
32,32,concert_singer,What are the names of all stadiums that did no...,SELECT name FROM stadium EXCEPT SELECT T2.name...,SELECT Name FROM stadium WHERE Stadium_ID NOT ...,False
35,35,concert_singer,List singer names and number of concerts for e...,"SELECT T2.name , count(*) FROM singer_in_conc...","SELECT s.Name, COUNT(si.concert_ID) AS Number_...",False
36,36,concert_singer,What are the names of the singers and number o...,"SELECT T2.name , count(*) FROM singer_in_conc...","SELECT s.Name, COUNT(si.concert_ID) AS Number_...",False
43,43,concert_singer,Find the number of concerts happened in the st...,SELECT count(*) FROM concert AS T1 JOIN stadiu...,SELECT COUNT(*) \nFROM concert \nJOIN stadium ...,False
44,44,concert_singer,What are the number of concerts that occurred ...,SELECT count(*) FROM concert AS T1 JOIN stadiu...,SELECT COUNT(*) \nFROM concert \nJOIN stadium ...,False
49,49,pets_1,Find the maximum weight for each type of pet. ...,"SELECT max(weight) , petType FROM pets GROUP ...","SELECT Pets.PetType, MAX(Pets.weight) AS MaxWe...",False


In [104]:
import shutil

shutil.make_archive(
    "/kaggle/working/zero_shot_results",
    "zip",
    "/kaggle/working",
    "zero_shot_results.csv"
)

print("ZIP created:")
print("/kaggle/working/zero_shot_results.zip")

ZIP creato:
/kaggle/working/zero_shot_results.zip


We look at the accuracy for the most common SQl structures in the databses.

In [8]:
structures = {
    "JOIN": r"\bJOIN\b",
    "WHERE": r"\bWHERE\b",
    "GROUP BY": r"\bGROUP\s+BY\b",
    "ORDER BY": r"\bORDER\s+BY\b",
    "HAVING": r"\bHAVING\b",
    "COUNT": r"\bCOUNT\s*\(",
    "AVG": r"\bAVG\s*\("
}

print("=" * 70)
print("ACCURACY ZERO-SHOT FOR EACH SQL STRUCTURE")
print("=" * 70)

for name, pattern in structures.items():
    
    # Cerchiamo la struttura nel GOLD SQL
    mask = zero_df["gold_sql"].str.contains(
        pattern, 
        case=False, 
        regex=True, 
        na=False
    )
    
    subset = zero_df[mask]
    
    total = len(subset)
    correct = subset["correct"].sum()
    
    if total > 0:
        accuracy = correct / total * 100
    else:
        accuracy = 0
    
    print(
        f"{name:<10} | "
        f"Total: {total:>3} | "
        f"Correct: {correct:>3} | "
        f"Accuracy: {accuracy:>6.2f}%"
    )

ACCURACY ZERO-SHOT PER STRUTTURA SQL
JOIN       | Totale: 412 | Corrette: 232 | Accuracy:  56.31%
WHERE      | Totale: 491 | Corrette: 346 | Accuracy:  70.47%
GROUP BY   | Totale: 277 | Corrette: 142 | Accuracy:  51.26%
ORDER BY   | Totale: 241 | Corrette: 166 | Accuracy:  68.88%
HAVING     | Totale:  79 | Corrette:  43 | Accuracy:  54.43%
COUNT      | Totale: 412 | Corrette: 279 | Accuracy:  67.72%
AVG        | Totale:  68 | Corrette:  47 | Accuracy:  69.12%


## 6. Few-Shot Prompting
We now add a layer of complexity: we provide the model with a few examples (question-query pairs) directly in the prompt to help it better grasp the expected format and relationships. 
To ensure we supply highly informative examples, we analyze the **Training Set** and select 5 diverse examples covering fundamental SQL constructs (JOIN, GROUP BY, WHERE, COUNT, ORDER BY).

In [41]:
import json

TRAIN_PATH = "/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/train_spider.json"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print("Train examples:", len(train_data))
print("\nFirst example:")
print(train_data[0])

Numero esempi train: 7000

Primo esempio:
{'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question': 'How many heads of the departments are older than 56 ?', 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?'], 'sql': {'except': None, 'from': {'conds': [], 'table_units': [['table_unit', 1]]}, 'groupBy': [], 'having': [], 'intersect': None, 'limit': None, 'orderBy': [], 'select': [False, [[3, [0, [0, 0, False], None]]]], 'union': None, 'where': [[False, 3, [0, [0, 10, False], None], 56.0, None]]}}


In [42]:
print("Train:", len(train_data))
print("Dev:", len(dev_data))

print("\nExample fields:")
print(train_data[0].keys())

Train: 7000
Dev: 1034

Campi di un esempio:
dict_keys(['db_id', 'query', 'query_toks', 'query_toks_no_value', 'question', 'question_toks', 'sql'])


In [60]:
from collections import Counter

# Function to classify which SQL constructs (JOIN, GROUP BY, etc.) are present in a query
def sql_features(sql):
    sql_upper = sql.upper()

    return {
        "COUNT": "COUNT(" in sql_upper,
        "AVG": "AVG(" in sql_upper,
        "WHERE": "WHERE" in sql_upper,
        "ORDER BY": "ORDER BY" in sql_upper,
        "GROUP BY": "GROUP BY" in sql_upper,
        "HAVING": "HAVING" in sql_upper,
        "JOIN": " JOIN " in sql_upper,
        "SUBQUERY": "SELECT" in sql_upper[sql_upper.find("SELECT") + 6:],
    }

feature_counts = Counter()

for example in train_data:
    features = sql_features(example["query"])

    for feature, present in features.items():
        if present:
            feature_counts[feature] += 1

print("SQL structures in the traning set:\n")

for feature, count in feature_counts.most_common():
    print(f"{feature:10} : {count}")

Strutture SQL nel training set:

WHERE      : 3502
JOIN       : 2771
COUNT      : 2324
GROUP BY   : 1775
ORDER BY   : 1628
SUBQUERY   : 1019
AVG        : 494
HAVING     : 427


In [61]:
def find_diverse_examples(train_data, required_features, max_examples=10):
    candidates = []
    seen_dbs = set()

    for example in train_data:
        features = sql_features(example["query"])

        if all(features[f] for f in required_features):
            db_id = example["db_id"]

            if db_id not in seen_dbs:
                candidates.append(example)
                seen_dbs.add(db_id)

        if len(candidates) >= max_examples:
            break

    return candidates


for category, features in categories.items():

    print("\n" + "=" * 70)
    print(category)
    print("=" * 70)

    examples = find_diverse_examples(
        train_data,
        features,
        max_examples=5
    )

    for i, example in enumerate(examples, 1):
        print(f"\n--- {i} ---")
        print("DB:", example["db_id"])
        print("Question:", example["question"])
        print("SQL:", example["query"])


WHERE

--- 1 ---
DB: department_management
Question: How many heads of the departments are older than 56 ?
SQL: SELECT count(*) FROM head WHERE age  >  56

--- 2 ---
DB: farm
Question: What are the hosts of competitions whose theme is not "Aliens"?
SQL: SELECT Hosts FROM farm_competition WHERE Theme !=  'Aliens'

--- 3 ---
DB: student_assessment
Question: List the id of students who never attends courses?
SQL: SELECT student_id FROM students WHERE student_id NOT IN (SELECT student_id FROM student_course_attendance)

--- 4 ---
DB: bike_1
Question: Give me the dates when the max temperature was higher than 85.
SQL: SELECT date FROM weather WHERE max_temperature_f  >  85

--- 5 ---
DB: book_2
Question: What are the titles of the books whose writer is not "Elaine Lee"?
SQL: SELECT Title FROM book WHERE Writer != "Elaine Lee"

COUNT_WHERE

--- 1 ---
DB: department_management
Question: How many heads of the departments are older than 56 ?
SQL: SELECT count(*) FROM head WHERE age  >  56

---

In [65]:
def find_exact_example(train_data, db_id, question):
    for example in train_data:
        if (
            example["db_id"] == db_id
            and example["question"] == question
        ):
            return example

    return None

# We select 5 diverse examples from the training set to build our Few-Shot Prompt
few_shot_examples = [
    # 1. COUNT + WHERE
    find_exact_example(
        train_data,
        "bike_1",
        "How many stations does Mountain View city has?"
    ),

    # 2. WHERE
    find_exact_example(
        train_data,
        "book_2",
        'What are the titles of the books whose writer is not "Elaine Lee"?'
    ),

    # 3. ORDER BY
    find_exact_example(
        train_data,
        "farm",
        "List the total number of horses on farms in ascending order."
    ),

    # 4. GROUP BY + HAVING + COUNT
    find_exact_example(
        train_data,
        "farm",
        "Show the official names of the cities that have hosted more than one competition."
    ),

    # 5. JOIN
    find_exact_example(
        train_data,
        "book_2",
        "Show the title and publication dates of books."
    )
]


print("Few-Shot examples:", len(few_shot_examples))

for i, example in enumerate(few_shot_examples, 1):
    print("\n" + "=" * 60)
    print(f"Example {i}")

    if example is None:
        print("ERROR: example not found!")
    else:
        print("DB:", example["db_id"])
        print("Question:", example["question"])
        print("SQL:", example["query"])

Numero esempi Few-Shot: 5

Example 1
DB: bike_1
Question: How many stations does Mountain View city has?
SQL: SELECT COUNT(*) FROM station WHERE city  =  "Mountain View"

Example 2
DB: book_2
Question: What are the titles of the books whose writer is not "Elaine Lee"?
SQL: SELECT Title FROM book WHERE Writer != "Elaine Lee"

Example 3
DB: farm
Question: List the total number of horses on farms in ascending order.
SQL: SELECT Total_Horses FROM farm ORDER BY Total_Horses ASC

Example 4
DB: farm
Question: Show the official names of the cities that have hosted more than one competition.
SQL: SELECT T1.Official_Name FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID  =  T2.Host_city_ID GROUP BY T2.Host_city_ID HAVING COUNT(*)  >  1

Example 5
DB: book_2
Question: Show the title and publication dates of books.
SQL: SELECT T1.Title ,  T2.Publication_Date FROM book AS T1 JOIN publication AS T2 ON T1.Book_ID  =  T2.Book_ID


In [47]:
def build_few_shot_prompt(question, schema, examples):
    prompt = f"""You are an expert Text-to-SQL system.

Your task is to convert a natural language question into a SQL query.

Use only the tables and columns provided in the database schema.
Do not invent tables or columns.

Here are some examples of natural language questions and their corresponding SQL queries:

"""

    for i, example in enumerate(examples, 1):
        prompt += f"""Example {i}:
Question:
{example["question"]}

SQL:
{example["query"]}

"""

    prompt += f"""Database schema:
{schema}

Question:
{question}

Return only the SQL query, without explanations, markdown or code fences.
"""

    return prompt

First we test it on only 10 examples.

In [66]:
test_results_few_shot = []

start_time = time.time()

for i, example in enumerate(dev_data[:10]):

    db_id = example["db_id"]
    question = example["question"]
    gold_sql = example["query"]

    schema = get_database_schema(db_id)

    prompt = build_few_shot_prompt(
        question,
        schema,
        few_shot_examples
    )

    try:
        predicted_sql = generate_sql(prompt)
        predicted_sql = clean_sql(predicted_sql)

    except Exception as e:
        predicted_sql = ""
        print(f"\nError in the example {i}: {e}")

    correct = execution_accuracy(
        db_id,
        gold_sql,
        predicted_sql
    )

    test_results_few_shot.append({
        "index": i,
        "db_id": db_id,
        "question": question,
        "gold_sql": gold_sql,
        "predicted_sql": predicted_sql,
        "correct": correct
    })

    print(
        f"[{i+1}/10] "
        f"Correct: {correct} | "
        f"Question: {question}"
    )

elapsed = time.time() - start_time

accuracy = (
    sum(r["correct"] for r in test_results_few_shot)
    / len(test_results_few_shot)
)

print("\n==============================")
print("TEST FEW-SHOT COMPLETED")
print("==============================")
print(f"Evaluated examples: {len(test_results_few_shot)}")
print(f"Accuracy: {accuracy:.2%}")
print(f"Time: {elapsed/60:.2f} minutes")

[1/10] Correct: True | Question: How many singers do we have?
[2/10] Correct: True | Question: What is the total number of singers?
[3/10] Correct: True | Question: Show name, country, age for all singers ordered by age from the oldest to the youngest.
[4/10] Correct: True | Question: What are the names, countries, and ages for every singer in descending order of age?
[5/10] Correct: True | Question: What is the average, minimum, and maximum age of all singers from France?
[6/10] Correct: True | Question: What is the average, minimum, and maximum age for all French singers?
[7/10] Correct: True | Question: Show the name and the release year of the song by the youngest singer.
[8/10] Correct: False | Question: What are the names and release years for all the songs of the youngest singer?
[9/10] Correct: True | Question: What are all distinct countries where singers above age 20 are from?
[10/10] Correct: True | Question: What are  the different countries with singers above age 20?

TEST

In [67]:
# see the errors
for r in test_results_few_shot:
    if not r["correct"]:
        print("INDEX:", r["index"])
        print("\nQUESTION:")
        print(r["question"])

        print("\nGOLD SQL:")
        print(r["gold_sql"])

        print("\nPREDICTED SQL:")
        print(r["predicted_sql"])

INDEX: 7

QUESTION:
What are the names and release years for all the songs of the youngest singer?

GOLD SQL:
SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

PREDICTED SQL:
SELECT T1.Song_Name, T1.Song_release_year 
FROM singer AS T1 
JOIN singer_in_concert AS T2 ON T1.Singer_ID = T2.Singer_ID 
WHERE T1.Age = (SELECT MIN(Age) FROM singer);


Now we test the entire development test, saving progressively the results in a CSV file.

In [75]:
FEW_SHOT_RESULTS_PATH = "/kaggle/working/few_shot_results_v2.csv"

print(FEW_SHOT_RESULTS_PATH)

results_few_shot = []
start_index = 0

def save_few_shot_results(results, path=FEW_SHOT_RESULTS_PATH):
    df = pd.DataFrame(results)
    df.to_csv(path, index=False)
    return df

/kaggle/working/few_shot_results_v2.csv


In [76]:
import os
import pandas as pd

# To recover previous results
def load_previous_results(path=FEW_SHOT_RESULTS_PATH):

    if os.path.exists(path):
        df = pd.read_csv(path)

        if len(df) > 0:
            results = df.to_dict("records")
            last_index = int(df["index"].max())

            print(f"File found: {path}")
            print(f"Saved results: {len(results)}")
            print(f"Last saved index: {last_index}")

            return results, last_index + 1

    print("No previous results found.")
    return [], 0

In [77]:
results_few_shot, start_index = load_previous_results()

print("\nStart index:", start_index)
print("Results loaded:", len(results_few_shot))

Nessun risultato precedente trovato.

Start index: 0
Risultati caricati: 0


In [79]:
start_time = time.time()

for i in range(start_index, len(dev_data)):

    example = dev_data[i]

    db_id = example["db_id"]
    question = example["question"]
    gold_sql = example["query"]

    schema = get_database_schema(db_id)

    prompt = build_few_shot_prompt(
        question,
        schema,
        few_shot_examples
    )

    try:
        predicted_sql = generate_sql(prompt)
        predicted_sql = clean_sql(predicted_sql)

    except Exception as e:
        predicted_sql = ""
        print(f"\nErrore nell'esempio {i}: {e}")

    try:
        correct = execution_accuracy(
            db_id,
            gold_sql,
            predicted_sql
        )

    except Exception as e:
        correct = False
        print(f"\nErrore nella valutazione dell'esempio {i}: {e}")

    results_few_shot.append({
        "index": i,
        "db_id": db_id,
        "question": question,
        "gold_sql": gold_sql,
        "predicted_sql": predicted_sql,
        "correct": correct
    })

    if (i + 1) % 10 == 0:

        save_few_shot_results(results_few_shot)

        elapsed = time.time() - start_time

        accuracy = (
            sum(r["correct"] for r in results_few_shot)
            / len(results_few_shot)
        )

        print(
            f"[{i+1}/{len(dev_data)}] "
            f"Accuracy: {accuracy:.2%} | "
            f"Time: {elapsed/60:.1f} min"
        )

save_few_shot_results(results_few_shot)

final_accuracy = (
    sum(r["correct"] for r in results_few_shot)
    / len(results_few_shot)
)

elapsed = time.time() - start_time

print("\n==============================")
print("FEW-SHOT COMPLETED")
print("==============================")
print(f"Evaluated examples: {len(results_few_shot)}")
print(f"Final accuracy: {final_accuracy:.2%}")
print(f"Time: {elapsed/60:.2f} minutes")
print(f"Results saved in: {FEW_SHOT_RESULTS_PATH}")

[10/1034] Accuracy: 90.00% | Time: 0.5 min
[20/1034] Accuracy: 90.00% | Time: 0.9 min
[30/1034] Accuracy: 80.00% | Time: 1.5 min
[40/1034] Accuracy: 75.00% | Time: 2.2 min
[50/1034] Accuracy: 72.00% | Time: 2.7 min
[60/1034] Accuracy: 71.67% | Time: 3.4 min
[70/1034] Accuracy: 71.43% | Time: 4.1 min
[80/1034] Accuracy: 68.75% | Time: 4.6 min
[90/1034] Accuracy: 68.89% | Time: 5.2 min
[100/1034] Accuracy: 70.00% | Time: 5.9 min
[110/1034] Accuracy: 67.27% | Time: 6.6 min
[120/1034] Accuracy: 63.33% | Time: 7.2 min
[130/1034] Accuracy: 62.31% | Time: 7.8 min
[140/1034] Accuracy: 62.86% | Time: 8.3 min
[150/1034] Accuracy: 64.00% | Time: 8.8 min
[160/1034] Accuracy: 63.12% | Time: 9.5 min
[170/1034] Accuracy: 63.53% | Time: 10.1 min
[180/1034] Accuracy: 61.11% | Time: 10.8 min
[190/1034] Accuracy: 63.16% | Time: 11.1 min
[200/1034] Accuracy: 65.00% | Time: 11.5 min
[210/1034] Accuracy: 66.67% | Time: 11.9 min
[220/1034] Accuracy: 66.82% | Time: 12.5 min
[230/1034] Accuracy: 66.52% | Time:

In [2]:
import pandas as pd
import os

path = "/kaggle/working/few_shot_results_v2.csv"

df = pd.read_csv(path)

print("File exists:", os.path.exists(path))
print("Dimension:", os.path.getsize(path) / 1024, "KB")
print("Rows:", len(df))
print("First columns:", list(df.columns))
print("Accuracy:", df["correct"].mean())

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/few_shot_results_v2.csv'

In [84]:
import shutil

shutil.make_archive(
    "/kaggle/working/few_shot_results_v2",
    "zip",
    "/kaggle/working",
    "few_shot_results_v2.csv"
)

print("ZIP created:")
print("/kaggle/working/few_shot_results_v2.zip")

ZIP creato:
/kaggle/working/few_shot_results_v2.zip


Analyzing the errors.

In [85]:
correct = df["correct"].sum()
wrong = (~df["correct"]).sum()

print("Few-Shot results")
print("===================")
print(f"Total:    {len(df)}")
print(f"Correct:  {correct}")
print(f"Wrong:    {wrong}")
print(f"Accuracy:  {df['correct'].mean():.2%}")

Risultati Few-Shot
Totale:    1034
Corrette:  730
Errate:    304
Accuracy:  70.60%


In [86]:
# Select only the wrong prediction 
errors_few = df[~df["correct"]].copy()

print("Few-Shot errors:", len(errors_few))

# first 10 errors
for _, row in errors_few.head(10).iterrows():

    print("\n" + "=" * 80)

    print("INDEX:")
    print(row["index"])

    print("\nDATABASE:")
    print(row["db_id"])

    print("\nQUESTION:")
    print(row["question"])

    print("\nGOLD SQL:")
    print(row["gold_sql"])

    print("\nPREDICTED SQL:")
    print(row["predicted_sql"])

Errori Few-Shot: 304

INDEX:
7

DATABASE:
concert_singer

QUESTION:
What are the names and release years for all the songs of the youngest singer?

GOLD SQL:
SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

PREDICTED SQL:
SELECT T1.Song_Name, T1.Song_release_year 
FROM singer AS T1 
JOIN singer_in_concert AS T2 ON T1.Singer_ID = T2.Singer_ID 
WHERE T1.Age = (SELECT MIN(Age) FROM singer);

INDEX:
15

DATABASE:
concert_singer

QUESTION:
What are the locations and names of all stations with capacity between 5000 and 10000?

GOLD SQL:
SELECT LOCATION ,  name FROM stadium WHERE capacity BETWEEN 5000 AND 10000

PREDICTED SQL:
SELECT LOCATION, Name FROM station WHERE Capacity BETWEEN 5000 AND 10000

INDEX:
22

DATABASE:
concert_singer

QUESTION:
Show the stadium name and the number of concerts in each stadium.

GOLD SQL:
SELECT T2.name ,  count(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id GROUP BY T1.stadium_id

PREDICTED SQL:
SELECT T1.Na

In [87]:
import re

errors_few = df[~df["correct"]].copy()

features = [
    "JOIN",
    "WHERE",
    "GROUP BY",
    "ORDER BY",
    "HAVING",
    "COUNT",
    "AVG",
    "MAX",
    "MIN",
    "NOT IN",
    "EXCEPT"
]

print("STRUCTURES PRESENT IN THE FEW-SHOT ERRORS")
print("=" * 50)

for feature in features:
    gold_count = errors_few["gold_sql"].str.upper().str.contains(
        re.escape(feature),
        na=False
    ).sum()

    predicted_count = errors_few["predicted_sql"].str.upper().str.contains(
        re.escape(feature),
        na=False
    ).sum()

    print(
        f"{feature:10s} | "
        f"Gold: {gold_count:3d} | "
        f"Predicted: {predicted_count:3d}"
    )

STRUTTURE PRESENTI NEGLI ERRORI FEW-SHOT
JOIN       | Gold: 190 | Predicted: 176
WHERE      | Gold: 152 | Predicted: 160
GROUP BY   | Gold: 118 | Predicted: 120
ORDER BY   | Gold:  68 | Predicted:  60
HAVING     | Gold:  34 | Predicted:  39
COUNT      | Gold: 180 | Predicted: 174
AVG        | Gold:  15 | Predicted:  15
MAX        | Gold:  12 | Predicted:  18
MIN        | Gold:   9 | Predicted:   9
NOT IN     | Gold:   9 | Predicted:  14
EXCEPT     | Gold:  13 | Predicted:   1


Calculate the accuracy for each structures.

In [88]:
features = [
    "JOIN",
    "WHERE",
    "GROUP BY",
    "ORDER BY",
    "HAVING",
    "COUNT",
    "AVG",
    "MAX",
    "MIN",
    "NOT IN",
    "EXCEPT"
]

print("ACCURACY FEW-SHOT FOR SQL STRUCTURE")
print("=" * 70)

for feature in features:

    mask = df["gold_sql"].str.upper().str.contains(
        re.escape(feature),
        na=False
    )

    subset = df[mask]

    if len(subset) > 0:
        accuracy = subset["correct"].mean()

        print(
            f"{feature:10s} | "
            f"Total: {len(subset):3d} | "
            f"Correct: {subset['correct'].sum():3d} | "
            f"Accuracy: {accuracy:.2%}"
        )

ACCURACY FEW-SHOT PER STRUTTURA SQL
JOIN       | Totale: 412 | Corrette: 222 | Accuracy: 53.88%
WHERE      | Totale: 491 | Corrette: 339 | Accuracy: 69.04%
GROUP BY   | Totale: 277 | Corrette: 159 | Accuracy: 57.40%
ORDER BY   | Totale: 241 | Corrette: 173 | Accuracy: 71.78%
HAVING     | Totale:  79 | Corrette:  45 | Accuracy: 56.96%
COUNT      | Totale: 534 | Corrette: 354 | Accuracy: 66.29%
AVG        | Totale:  68 | Corrette:  53 | Accuracy: 77.94%
MAX        | Totale:  40 | Corrette:  28 | Accuracy: 70.00%
MIN        | Totale:  27 | Corrette:  18 | Accuracy: 66.67%
NOT IN     | Totale:  46 | Corrette:  37 | Accuracy: 80.43%
EXCEPT     | Totale:  31 | Corrette:  18 | Accuracy: 58.06%


## 7. Results Comparison: Zero-Shot vs Few-Shot
We load the results generated on the entire Dev Set (1034 queries) for both approaches.
To verify if the slight improvement brought by the Few-Shot prompt is statistically significant or just random chance, we run **McNemar's Test** (which is ideal for comparing the performance of two paired classifiers on the same dataset).

In [4]:
import pandas as pd

# Loading both files
zero_path = "/kaggle/input/datasets/caterinavespa/results/zero_shot_results/zero_shot_results.csv"
few_path = "/kaggle/input/datasets/caterinavespa/results/few_shot_results_v2/few_shot_results_v2.csv"

zero_df = pd.read_csv(zero_path)
few_df = pd.read_csv(few_path)

print("ZERO-SHOT")
print("Rows:", len(zero_df))
print("Columns:", list(zero_df.columns))

print("\nFEW-SHOT")
print("Rows:", len(few_df))
print("Columns:", list(few_df.columns))

ZERO-SHOT
Righe: 1034
Colonne: ['index', 'db_id', 'question', 'gold_sql', 'predicted_sql', 'correct']

FEW-SHOT
Righe: 1034
Colonne: ['index', 'db_id', 'question', 'gold_sql', 'predicted_sql', 'correct']


In [5]:
zero_correct = zero_df["correct"].sum()
few_correct = few_df["correct"].sum()

zero_total = len(zero_df)
few_total = len(few_df)

zero_accuracy = zero_df["correct"].mean()
few_accuracy = few_df["correct"].mean()

print("=" * 60)
print("COMPARISON ZERO-SHOT vs FEW-SHOT")
print("=" * 60)

print(f"\nZERO-SHOT")
print(f"  Correct: {zero_correct}/{zero_total}")
print(f"  Wrong:   {zero_total - zero_correct}/{zero_total}")
print(f"  Accuracy: {zero_accuracy:.2%}")

print(f"\nFEW-SHOT")
print(f"  Correct: {few_correct}/{few_total}")
print(f"  Wrong:   {few_total - few_correct}/{few_total}")
print(f"  Accuracy: {few_accuracy:.2%}")

print("\n" + "-" * 60)

improvement = few_accuracy - zero_accuracy

print(f"Few-Shot Improvement: {improvement:+.2%}")
print(f"Difference: {improvement * 100:+.2f} percentage points")

CONFRONTO ZERO-SHOT vs FEW-SHOT

ZERO-SHOT
  Corrette: 726/1034
  Errate:   308/1034
  Accuracy: 70.21%

FEW-SHOT
  Corrette: 730/1034
  Errate:   304/1034
  Accuracy: 70.60%

------------------------------------------------------------
Miglioramento Few-Shot: +0.39%
Differenza: +0.39 punti percentuali


In [6]:
print("Stesso numero di righe:", len(zero_df) == len(few_df))

# Comparison of predictions (4 cases)
both_correct = ((zero_df["correct"] == True) & 
                (few_df["correct"] == True)).sum()

zero_wrong_few_correct = ((zero_df["correct"] == False) & 
                          (few_df["correct"] == True)).sum()

zero_correct_few_wrong = ((zero_df["correct"] == True) & 
                          (few_df["correct"] == False)).sum()

both_wrong = ((zero_df["correct"] == False) & 
              (few_df["correct"] == False)).sum()

print("\n" + "=" * 60)
print("DETAILED COMPARISON")
print("=" * 60)

print(f"\nBoth correct:              {both_correct}")
print(f"Zero-shot wrong → Few-shot correct: {zero_wrong_few_correct}")
print(f"Zero-shot correct → Few-shot wrong: {zero_correct_few_wrong}")
print(f"Both wrong:                {both_wrong}")

print("\nTotal check:")
print(both_correct + zero_wrong_few_correct + 
      zero_correct_few_wrong + both_wrong)

Stesso numero di righe: True

CONFRONTO DETTAGLIATO

Entrambi corretti:              669
Zero-shot errato → Few-shot corretto: 61
Zero-shot corretto → Few-shot errato: 57
Entrambi errati:                247

Controllo totale:
1034


Eseguiamo il **McNemar Test** per controllare se il miglioramento del 0,39% del few-shot sia effettivamente significativo o puro caso.

In [9]:
from statsmodels.stats.contingency_tables import mcnemar

# Build the contingency table (2x2 Confusion Matrix) for cross-errors
both_correct = ((zero_df["correct"] == True) & 
                (few_df["correct"] == True)).sum()

zero_correct_few_wrong = ((zero_df["correct"] == True) & 
                          (few_df["correct"] == False)).sum()

zero_wrong_few_correct = ((zero_df["correct"] == False) & 
                          (few_df["correct"] == True)).sum()

both_wrong = ((zero_df["correct"] == False) & 
              (few_df["correct"] == False)).sum()

table = [
    [both_correct, zero_correct_few_wrong],
    [zero_wrong_few_correct, both_wrong]
]

print("ontingency table:")
print()
print("                    Few-shot")
print("                  Correct   Wrong")
print(f"Zero-shot correct    {both_correct:4d}     {zero_correct_few_wrong:4d}")
print(f"Zero-shot wrong      {zero_wrong_few_correct:4d}     {both_wrong:4d}")

# The p-value will tell us if the performance difference is statistically significant (p < 0.05)
result = mcnemar(table, exact=True)

print("\n" + "=" * 60)
print("McNEMAR TEST")
print("=" * 60)

print(f"Statistic: {result.statistic}")
print(f"p-value:   {result.pvalue}")

if result.pvalue < 0.05:
    print("\nRisultato: DIFFERENCE STATISTICALLY SIGNIFICANT")
else:
    print("\nRisultato: DIFFERENCE NOT STATISTICALLY SIGNIFICANT")

Tabella di contingenza:

                    Few-shot
                  Corretto   Errato
Zero-shot corretto     669       57
Zero-shot errato        61      247

McNEMAR TEST
Statistic: 57.0
p-value:   0.7825558614117515

Risultato: DIFFERENZA NON STATISTICAMENTE SIGNIFICATIVA


## 8. Error Analysis
In addition to quantitative validation, we perform a qualitative inspection of the errors.
We isolate the queries where both methods (Zero and Few-Shot) failed to identify the model's main weak points. We also look at "regression" cases—queries where the model guessed correctly in Zero-Shot but actually failed once given the Few-Shot prompt.

In [10]:
# Cases in which both methods are incorrect
both_wrong_df = zero_df[
    (zero_df["correct"] == False) &
    (few_df["correct"] == False)
].copy()

# Add the few-shot prediction
both_wrong_df["few_predicted_sql"] = few_df.loc[
    both_wrong_df.index, "predicted_sql"
]

print("Cases in which both are wrong:", len(both_wrong_df))

Casi in cui entrambi sbagliano: 247


In [11]:
import re

structures = {
    "JOIN": r"\bJOIN\b",
    "WHERE": r"\bWHERE\b",
    "GROUP BY": r"\bGROUP\s+BY\b",
    "ORDER BY": r"\bORDER\s+BY\b",
    "HAVING": r"\bHAVING\b",
    "COUNT": r"\bCOUNT\s*\(",
    "AVG": r"\bAVG\s*\(",
    "MAX": r"\bMAX\s*\(",
    "MIN": r"\bMIN\s*\(",
    "SUM": r"\bSUM\s*\(",
    "INTERSECT": r"\bINTERSECT\b",
    "UNION": r"\bUNION\b",
    "EXCEPT": r"\bEXCEPT\b"
}

print("=" * 75)
print("STRUCTURES IN THE 247 WRONG CASES")
print("=" * 75)

for name, pattern in structures.items():

    mask = both_wrong_df["gold_sql"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )

    count = mask.sum()
    percentage = count / len(both_wrong_df) * 100

    print(
        f"{name:<12} | "
        f"Cases: {count:>3} | "
        f"Percentage: {percentage:>6.2f}%"
    )

STRUTTURE PRESENTI NEI 247 CASI SBAGLIATI DA ENTRAMBI
JOIN         | Casi: 146 | Percentuale:  59.11%
WHERE        | Casi: 124 | Percentuale:  50.20%
GROUP BY     | Casi: 102 | Percentuale:  41.30%
ORDER BY     | Casi:  57 | Percentuale:  23.08%
HAVING       | Casi:  26 | Percentuale:  10.53%
COUNT        | Casi: 103 | Percentuale:  41.70%
AVG          | Casi:  15 | Percentuale:   6.07%
MAX          | Casi:  11 | Percentuale:   4.45%
MIN          | Casi:   8 | Percentuale:   3.24%
SUM          | Casi:  14 | Percentuale:   5.67%
INTERSECT    | Casi:  23 | Percentuale:   9.31%
UNION        | Casi:   9 | Percentuale:   3.64%
EXCEPT       | Casi:  11 | Percentuale:   4.45%


In [12]:
# Show 20 random cases where both are wrong

sample_wrong = both_wrong_df.sample(
    n=min(20, len(both_wrong_df)),
    random_state=42
)

for i, (_, row) in enumerate(sample_wrong.iterrows(), 1):

    print("\n" + "=" * 90)
    print(f"CASE {i}")
    print("=" * 90)

    print(f"INDEX: {row['index']}")
    print(f"DB: {row['db_id']}")

    print("\nQUESTION:")
    print(row["question"])

    print("\nGOLD SQL:")
    print(row["gold_sql"])

    print("\nZERO-SHOT:")
    print(row["predicted_sql"])

    print("\nFEW-SHOT:")
    print(row["few_predicted_sql"])


CASO 1
INDICE: 122
DB: car_1

DOMANDA:
What are the makers and models?

GOLD SQL:
SELECT Maker ,  Model FROM MODEL_LIST;

ZERO-SHOT:
SELECT T1.Maker, T2.Model FROM car_makers AS T1 JOIN model_list AS T2 ON T1.Id = T2.Maker;

FEW-SHOT:
SELECT T1.Maker, T2.Model FROM car_makers AS T1 JOIN model_list AS T2 ON T1.Id = T2.Maker

CASO 2
INDICE: 43
DB: concert_singer

DOMANDA:
Find the number of concerts happened in the stadium with the highest capacity.

GOLD SQL:
SELECT count(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id ORDER BY T2.Capacity DESC LIMIT 1

ZERO-SHOT:
SELECT COUNT(*) 
FROM concert 
JOIN stadium ON concert.Stadium_ID = stadium.Stadium_ID 
WHERE stadium.Capacity = (SELECT MAX(Capacity) FROM stadium);

FEW-SHOT:
SELECT COUNT(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.Stadium_ID = T2.Stadium_ID WHERE T2.Capacity = (SELECT MAX(Capacity) FROM stadium)

CASO 3
INDICE: 779
DB: world_1

DOMANDA:
Return the country codes for countries that do not spea

In [6]:
# Cases in which:
# Zero-shot = correct
# Few-shot = wrong

regressed = zero_df[
    (zero_df["correct"] == True) &
    (few_df["correct"] == False)
].copy()

# Aggiungiamo la predizione few-shot
regressed["few_predicted_sql"] = few_df.loc[
    regressed.index, "predicted_sql"
]

print("Regressed cases", len(regressed))

Numero di casi peggiorati: 57


In [7]:
# 20 random cases among regressed cases
sample_regressed = regressed.sample(
    n=min(20, len(regressed)),
    random_state=42
)

for i, (_, row) in enumerate(sample_regressed.iterrows(), 1):

    print("\n" + "=" * 90)
    print(f"CASE {i}")
    print("=" * 90)

    print(f"INDEX: {row['index']}")
    print(f"DB: {row['db_id']}")

    print("\nQUESTION:")
    print(row["question"])

    print("\nGOLD SQL:")
    print(row["gold_sql"])

    print("\nZERO-SHOT (CORRECT):")
    print(row["predicted_sql"])

    print("\nFEW-SHOT (WRONG):")
    print(row["few_predicted_sql"])


CASO 1
INDICE: 7
DB: concert_singer

DOMANDA:
What are the names and release years for all the songs of the youngest singer?

GOLD SQL:
SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

ZERO-SHOT (CORRETTO):
SELECT Song_Name, Song_release_year 
FROM singer 
ORDER BY Age ASC 
LIMIT 1;

FEW-SHOT (ERRATO):
SELECT T1.Song_Name, T1.Song_release_year 
FROM singer AS T1 
JOIN singer_in_concert AS T2 ON T1.Singer_ID = T2.Singer_ID 
WHERE T1.Age = (SELECT MIN(Age) FROM singer);

CASO 2
INDICE: 42
DB: concert_singer

DOMANDA:
What are the names and locations of the stadiums that had concerts that occurred in both 2014 and 2015?

GOLD SQL:
SELECT T2.name ,  T2.location FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.Year  =  2014 INTERSECT SELECT T2.name ,  T2.location FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.Year  =  2015

ZERO-SHOT (CORRETTO):
SELECT s.Name, s.Location
FROM stadium s
JOIN concert c